# Backend comparison simulation

Compares Riesz representer learners (rieszboost with XGBoost trees, and riesznet) for the ATE and ATT. Each replicate cross-fits the outcome regression and the representer, tunes both by `GridSearchCV`, and plugs them into the one-step estimator. The learners, tuning grids, and estimators live in `run_backend_comparison.py`; the DGPs live in `dgp.py` (hard: 40 covariates) and `run_easy_dgp.py` (easy: 10 covariates). This notebook runs one of them and summarizes the results.

**Setup on the server**

1. Clone the repo and copy the `examples/backend_comparison/` folder into it (keep this notebook inside that folder).
2. Install the packages into the notebook's kernel environment, e.g. from the repo root:
   `pip install -e packages/rieszreg/python -e packages/rieszboost/python -e packages/riesznet/python xgboost torch scikit-learn pandas joblib scipy`
   (the path cell below also puts the packages on the path directly, so an install is only needed for their dependencies).
3. Run the cells top to bottom. Restart the kernel before re-running the first cell.

Results are appended to the CSV after every replicate, so if the kernel dies or the connection drops, re-running the notebook skips the finished replicates and continues.

In [ ]:
# One thread per process: the replicates already run in parallel, and torch
# (riesznet) and xgboost (rieszboost) can crash on macOS / deadlock when both
# use multithreaded OpenMP. Must run before either library is imported.
import os
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[var] = "1"

In [ ]:
# Put this folder and the workspace packages on the path. PYTHONPATH is set
# too so the parallel worker processes inherit it.
import sys
from pathlib import Path

HERE = Path.cwd()                       # examples/backend_comparison
REPO = HERE.parents[1]
PATHS = [str(HERE)] + sorted(str(p) for p in REPO.glob("packages/*/python"))
sys.path[:0] = PATHS
os.environ["PYTHONPATH"] = os.pathsep.join(PATHS + [os.environ.get("PYTHONPATH", "")])

## Settings

In [ ]:
DGP = "easy"          # "easy" (run_easy_dgp.py) or "hard" (dgp.py)
N_REPS = 20
N_DATA = 2000
CV_FOLDS = 5          # GridSearchCV folds for tuning mu and alpha
N_JOBS = 8            # replicates run in parallel; at most the number of cores
OUT = HERE / f"results_{DGP}_n{N_DATA}"   # writes OUT_results.csv and OUT_fits.csv

In [ ]:
import importlib
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

sim = importlib.import_module({"easy": "run_easy_dgp", "hard": "run_backend_comparison"}[DGP])
cfg = importlib.import_module("run_backend_comparison")
print("rieszboost grid:", cfg.RIESZBOOST_GRID)
print("riesznet learning rates:", cfg.NET_LEARNING_RATES)
print("outcome regression grid:", cfg.OUTCOME_GRID)

## True estimands (Monte Carlo)

In [ ]:
true_psi = {
    name: dgp_cls.true_psi(np.random.default_rng(12345), cfg.N_MC_TRUTH)
    for name, (dgp_cls, _, _) in sim.ESTIMANDS.items()
}
true_psi

## Run

Each replicate also records, for every tuned fit, the selected hyperparameters and the early-stopping iteration. Re-running this cell continues from the replicates already saved in `OUT_results.csv`.

In [ ]:
def one_rep(rep, n, cv_folds):
    """Runs sim.run_rep, logging each GridSearchCV's selected hyperparameters.
    The logging subclass is defined here so it is built inside the worker."""
    import time
    from sklearn.model_selection import GridSearchCV

    log = []

    class LoggedGridSearchCV(GridSearchCV):
        def fit(self, X, y=None, **kwargs):
            start = time.time()
            super().fit(X, y, **kwargs)
            fitted = getattr(self.best_estimator_, "best_", self.best_estimator_)   # riesznet: chosen restart
            log.append({
                "rep": rep,
                "model": type(fitted).__name__,
                "best_params": str({k.replace("estimator__", ""): v for k, v in self.best_params_.items()}),
                "best_iteration": getattr(fitted, "best_iteration_", None),
                "seconds": time.time() - start,
            })
            return self

    sim.GridSearchCV = LoggedGridSearchCV
    return sim.run_rep(rep, n=n, cv_folds=cv_folds), log


results_csv, fits_csv = Path(f"{OUT}_results.csv"), Path(f"{OUT}_fits.csv")
done = set(pd.read_csv(results_csv).rep) if results_csv.exists() else set()
todo = [rep for rep in range(N_REPS) if rep not in done]
print(f"{len(done)} replicates already saved, running {len(todo)}")

start = time.time()
jobs = Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(
    delayed(one_rep)(rep, N_DATA, CV_FOLDS) for rep in todo
)
for k, (rows, log) in enumerate(jobs, 1):
    rows = pd.DataFrame(rows).assign(truth=lambda d: d.estimand.map(true_psi))
    rows.to_csv(results_csv, mode="a", header=not results_csv.exists(), index=False)
    pd.DataFrame(log).to_csv(fits_csv, mode="a", header=not fits_csv.exists(), index=False)
    print(f"rep {rows.rep.iloc[0]} done ({k}/{len(todo)}, {(time.time() - start) / 60:.1f} min)", flush=True)

## Summary

In [ ]:
results = pd.read_csv(results_csv)
fits = pd.read_csv(fits_csv)
print(f"{results.rep.nunique()} replicates")
sim.summarize(results.copy()).round(3)

Monte Carlo standard error of the bias, and the replicates with the largest representer error (to spot runaway fits):

In [ ]:
error = results.est - results.truth
by = error.groupby([results.estimand, results.learner])
display(pd.DataFrame({"bias": by.mean(), "mc_se": by.std() / np.sqrt(by.size())}).round(3))
results.assign(error=error).nlargest(5, "alpha_rmse")[["rep", "estimand", "learner", "est", "error", "se", "alpha_rmse"]].round(3)

## Selected hyperparameters

In [ ]:
riesz_fits = fits[fits.model != "XGBRegressorES"]
display(riesz_fits.groupby("model").best_params.value_counts().to_frame("n_fits"))
riesz_fits.groupby("model")[["best_iteration", "seconds"]].describe().round(0)